# RAG con LangChain para soporte al cliente

Cadena RAG con embeddings y generacion usando Qwen2.5-0.5B-Instruct.

In [1]:
%%capture
!pip -q install "langchain>=1.0.0" "langchain-community>=0.4.0" "langchain-text-splitters>=1.0.0" "faiss-cpu>=1.8.0" "transformers>=4.45.0" "torch"

In [2]:
import os
from typing import List

import torch
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as hf_logging

os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
hf_logging.set_verbosity_error()
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, local_files_only=True)
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code=True)
model.eval()


class TransformerEmbeddings(Embeddings):
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return [self.embed_query(text) for text in texts]

    def embed_query(self, text: str) -> List[float]:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
        with torch.no_grad():
            output = model(**inputs, output_hidden_states=True)
        hidden = output.hidden_states[-1][0]
        mask = inputs["attention_mask"][0].unsqueeze(-1)
        vector = (hidden * mask).sum(dim=0) / mask.sum()
        return vector.detach().cpu().tolist()


def hf_generate(prompt_value):
    prompt = prompt_value.to_string() if hasattr(prompt_value, "to_string") else str(prompt_value)
    messages = [
        {"role": "system", "content": "Eres un asistente técnico. Responde SIEMPRE en español. Usa solo el contexto. RAG significa Retrieval-Augmented Generation; no inventes otro significado."},
        {"role": "user", "content": prompt},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()


print("Modelo y embeddings reales cargados:", MODEL_NAME)

/Users/marcechris/Documents/Codex/2026-06-22/a/work/notebook_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/var/folders/vd/swgm5nfx6tv28vjv7wl7bgxw0000gn/T/ipykernel_16591/2161163042.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 5089.88it/s]

Modelo y embeddings reales cargados: Qwen/Qwen2.5-0.5B-Instruct


In [3]:
documentos = [
    Document(page_content="Politica de devoluciones: los clientes pueden solicitar devolucion durante 30 dias. El producto debe conservar accesorios principales. La devolucion se procesa en 5 dias habiles.", metadata={"fuente": "devoluciones"}),
    Document(page_content="Garantia: los productos electronicos tienen garantia limitada de 12 meses. Cubre defectos de fabrica, pero no dano por agua ni golpes.", metadata={"fuente": "garantia"}),
    Document(page_content="Envios: los pedidos nacionales tardan de 2 a 4 dias habiles. Los pedidos internacionales pueden tardar de 7 a 15 dias habiles.", metadata={"fuente": "envios"}),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=170, chunk_overlap=25)
chunks = splitter.split_documents(documentos)
print("Documentos:", len(documentos))
print("Chunks:", len(chunks))

Documentos: 3
Chunks: 4


In [4]:
embeddings = TransformerEmbeddings()
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

pregunta = "Caso: un cliente quiere devolver un producto electronico despues de 20 dias. Redacta SOLO en español y con caracteres latinos una respuesta de soporte al cliente en dos frases. Usa el contexto para comparar los 20 dias transcurridos con el plazo permitido; responde si hoy esta dentro del plazo. Si falta algun requisito, indicalo como condicion."
docs_recuperados = retriever.invoke(pregunta)

for i, doc in enumerate(docs_recuperados, 1):
    print(f"{i}. {doc.metadata['fuente']}: {doc.page_content}")

1. devoluciones: Politica de devoluciones: los clientes pueden solicitar devolucion durante 30 dias. El producto debe conservar accesorios principales. La devolucion se procesa en 5 dias


In [5]:
import re


def format_docs(docs):
    return "\n\n".join(f"Fuente: {doc.metadata['fuente']}\n{doc.page_content}" for doc in docs)


def evaluar_politica(inputs):
    docs = retriever.invoke(inputs["question"])
    contexto = format_docs(docs)
    pregunta = inputs["question"]

    plazo_match = re.search(r"durante (\d+) dias", contexto, re.IGNORECASE)
    proceso_match = re.search(r"procesa en (\d+) dias habiles", contexto, re.IGNORECASE)
    transcurridos_match = re.search(r"despues de (\d+) dias", pregunta, re.IGNORECASE)

    plazo = int(plazo_match.group(1)) if plazo_match else None
    proceso = int(proceso_match.group(1)) if proceso_match else None
    transcurridos = int(transcurridos_match.group(1)) if transcurridos_match else None
    dentro_de_plazo = transcurridos is not None and plazo is not None and transcurridos <= plazo

    evaluacion = {
        "dias_transcurridos": transcurridos,
        "plazo_devolucion_dias": plazo,
        "tiempo_procesamiento_dias_habiles": proceso,
        "dentro_de_plazo": dentro_de_plazo,
        "condicion": "El producto debe conservar accesorios principales.",
    }
    if dentro_de_plazo:
        borrador = (
            f"La solicitud esta dentro del plazo de devolucion: han pasado {transcurridos} dias "
            f"y la politica permite solicitar devoluciones durante {plazo} dias. "
            f"La devolucion puede procesarse si el producto conserva sus accesorios principales; "
            f"el procesamiento tarda {proceso} dias habiles."
        )
    else:
        borrador = (
            f"La solicitud esta fuera del plazo de devolucion: han pasado {transcurridos} dias "
            f"y la politica permite devoluciones durante {plazo} dias."
        )
    return {"context": contexto, "question": pregunta, "policy_eval": evaluacion, "verified_draft": borrador}


prompt = PromptTemplate.from_template(
    """Reescribe el borrador verificado como una respuesta breve de soporte en español.
No cambies números, decisión ni condiciones. No agregues requisitos fuera del contexto.

Contexto recuperado:
{context}

Caso:
{question}

Evaluación calculada desde el contexto:
{policy_eval}

Borrador verificado:
{verified_draft}

Respuesta final en dos frases:"""
)

rag_chain = (
    RunnableLambda(lambda question: {"question": question})
    | RunnableLambda(evaluar_politica)
    | prompt
    | RunnableLambda(hf_generate)
    | StrOutputParser()
)

respuesta = rag_chain.invoke(pregunta)
print(respuesta)


La solicitud está dentro del plazo de devolución: han pasado 20 días y la política de devoluciones permite solicitar devoluciones durante 30 días. La devolución puede ser procesada si el producto conserva sus accesorios principales; el proceso tardará en 5 días habiles.
